In [1]:
ratpup           <- read.table('data/rat_pup.dat', header=TRUE)
ratpup$sex       <- as.factor(ratpup$sex)
ratpup$litter    <- as.factor(ratpup$litter)
ratpup$treatment <- as.factor(ratpup$treatment)

# Building the Model III
In our third model building step, we need to write the models for the next level in the hierarchy. Depending upon whether we decided to treat certain terms as *fixed* or *random* will change how these higher-level models are expressed.

## The Consequences of Random Terms
In the previous step, we determined whether to allow certain terms to vary per-litter, or whether we want to treat them as global effects *irrespective* of litter. What we were actually doing here was deciding whether a term was to be treated as *random* or *fixed*. Because we decided to treat *both* $\mu^{(k)}$ and $\alpha_{j}^{(k)}$ as *varying per-litter*, then these are *both* random variables. For every new litter we sample, their values will change. This means that these terms must have associated probability distributions that describe their behaviour over repeated sampling. 

For simplicity, if we assume these distributions are normal, our model is currently

$$
\begin{alignat*}{1}
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij} \\
    \eta^{(k)}_{ij}  &\sim \mathcal{N}(0, \sigma^{2}_{1}) \\
    \mu^{(k)}        &\sim \mathcal{N}(\mu, \sigma^{2}_{2}) \\
    \alpha^{(k)}_{j} &\sim \mathcal{N}(\alpha_{j}, \sigma^{2}_{3})
\end{alignat*}
$$

The errors are always random, so this had not changed. What is now different is that we give both $\mu^{(k)}$ and $\alpha^{(k)}_{j}$ probability distributions. So, the average weight of each litter $\mu^{(k)}$ is conceptualised as being drawn from a larger population of average weights, with some population mean weight of $\mu$. Similarly, the effect of `sex` within each litter $\alpha^{(k)}_{j}$ is conceptualised as being drawn from a larger population of `sex` effects, with some population sex effect of $\alpha_{j}$.

Remembering that we can always write probability models in terms of expressions for the means plus random error, we can also rewrite the above as

$$
\begin{alignat*}{1}
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij} \\
    \mu^{(k)}        &= \mu + \xi^{(k)} \\
    \alpha^{(k)}_{j} &= \alpha_{j} + \phi^{(k)}_{j}
\end{alignat*}
$$

So, now we have two extra error terms in the form of $\xi^{(k)}$ and $\phi^{(k)}_{j}$. The assumptions about all the error terms are then

$$
\begin{alignat*}{1}
    \eta^{(k)}_{ij} &\sim \mathcal{N}(0,\sigma^{2}_{1}) \\
    \xi^{(k)}       &\sim \mathcal{N}(0,\sigma^{2}_{2}) \\
    \phi^{(k)}_{j}  &\sim \mathcal{N}(0,\sigma^{2}_{3})
\end{alignat*}
$$

So we can see that what we have ended up with is a model containing *three* variances. We have partitioned the overall variance of the data into 3 components

$$
\text{Var}\left(y\right) = \sigma^{2} = \sigma^{2}_{1} + \sigma^{2}_{2} + \sigma^{2}_{3}.
$$

Notice that we did not *start* by doing this. This is simply where we have ended-up by building the model slowly in a principled fashion. This structure has simply *fallen-out* of the decisions we have made along the way. So, our current conceptualisation is of a two-level model of the following form

$$
\begin{alignat*}{2}
    \text{Level 1} \\
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij} &\quad\small{\text{Weight of pup $i$ of sex $j$ in litter $k$}} \\
    \text{Level 2} \\
    \mu^{(k)}        &= \mu + \xi^{(k)} &\quad\small{\text{Average weight of litter $k$}} \\
    \alpha^{(k)}_{j} &= \alpha_{j} + \phi^{(k)}_{j} &\quad\small{\text{Effect of sex $j$ in litter $k$}}
\end{alignat*}
$$

However, there is more we can now do with the two 2nd-level models.

## Modelling $\mu^{(k)}$
Let us start by focussing on the 2nd-level model of $\mu^{(k)}$. If we treat it like any other regression model, we can decide whether there is more we can include to model the mean weight of litter, beyond just the population mean $\mu$. So, let us pretend we *only* have the model

$$
\mu^{(k)} = \mu + \xi^{(k)}.
$$

Is there anything else in this dataset that we think might relate to the mean weight of each litter? At this level, we are modelling litters *as a whole*, so we need to consider any variables that are constant *within* a litter but change *across* litters. Back in step I, we examined several of these when we looked at only a single litter: `litsize`, `treatment` and `litter` itself. For the sake of simplicity, we will focus on `treatment` as this is of primary interest in this experiment[^litsize-foot]. So, an additional element that might affect the average weight in litter $k$ is the `treatment` that litter were given. If we denote the effect of the `treatment` that has been applied $\beta_{l}$, then our model of the mean becomes

$$
\mu^{(k)} = \mu + \beta_{l} + \xi^{(k)}.
$$

So, we think of the average weight of litter $k$ as depending upon the average weight of the population of litters $\mu$, the effect of `treatment` $l$ on litter weight $\beta_{l}$ and a litter-specific random error $\xi^{(k)}$.

## Modelling $\alpha^{(k)}_{j}$
Now, let us focus on the 2nd-level model of $\alpha^{(k)}_{j}$. Again, if we treat this like any other regression model, we can decide whether there is more we can include to model the effect of `sex` within each litter, beyond just the population effect $\alpha_{j}$. So, let us pretend we *only* have the model

$$
\alpha^{(k)}_{j} = \alpha_{j} + \phi^{(k)}_{j}
$$

Could any of the variables identified above relate to the effect of `sex` on the average weight of a litter? Again, we might want to consider the `treatment` that was applied. If the `treatment` affects the weight of male and female pups in a different way, then there *could* be a `treatment:sex` interaction. In reality, inclusion of this effect would depend upon what `treatment` actually does and whether there is cause to consider an interaction. However, for the sake of example, we can expand this model to

$$
\alpha^{(k)}_{j} = \alpha_{j} + (\alpha\beta)_{jl} + \phi^{(k)}_{j}
$$

So, this model is now saying that the effect of sex in litter $k$ is governed by the population effect of `sex` $\alpha_{j}$, the interaction between `treatment` and `sex` $(\alpha\gamma)_{jl}$, and a litter-specific random error $\phi^{(k)}_{j}$.

## The Final Model
Based on everything above, we arrive at the following model

$$
\begin{alignat*}{1}
    \text{Level 1} \\
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij}  \\
    \quad \\
    \text{Level 2} \\
    \mu^{(k)}        &= \mu + \beta_{l} + \xi^{(k)}  \\
    \alpha^{(k)}_{j} &= \alpha_{j} + (\alpha\beta)_{jl} + \phi^{(k)}_{j}
\end{alignat*}
$$

which we can perhaps make a bit clearer by labelling the terms explicitly

$$
\begin{alignat*}{1}
    \text{Level 1} \\
    \text{weight}^{(k)}_{ij} &= \text{mean}^{(k)} + \text{sex}^{(k)}_{j} + \eta^{(k)}_{ij}  \\
    \quad \\
    \text{Level 2} \\
    \text{mean}^{(k)}    &= \text{mean} + \text{treatment}_{l} + \xi^{(k)}  \\
    \text{sex}^{(k)}_{j} &= \text{sex}_{j} + (\text{sex} \times \text{treatment})_{jl} + \phi^{(k)}_{j}
\end{alignat*}
$$

## What if `sex` Were Fixed?
By way of comparison, let us see where we end up if we choose to make `sex` a *fixed-effect* rather than a *random-effect*. In this case, we do not give `sex` a distribution. Instead, we simply equate the litter-specific effect of `sex` to its population value. Going back to our representation from the beginning of this part, this gives

$$
\begin{alignat*}{1}
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij} \\
    \eta^{(k)}_{ij}  &\sim \mathcal{N}(0, \sigma^{2}_{1}) \\
    \mu^{(k)}        &\sim \mathcal{N}(\mu, \sigma^{2}_{2}) \\
    \alpha^{(k)}_{j} &= \alpha_{j}.
\end{alignat*}
$$

So now, both $\eta^{(k)}_{ij}$ and $\mu^{(k)}$ are *random*, but the effect of `sex` in litter $k$ $\left(\alpha^{(k)}_{j}\right)$ is *fixed* to a constant value. The consequence is that there are now only *two* variances, rather than *three*, because there is no variation associated with the effect of `sex`.

If we re-express the distributions in terms of mean + error, we now arrive at

$$
\begin{alignat*}{2}
    \text{Level 1} \\
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij} &\quad\small{\text{Weight of pup $i$ of sex $j$ in litter $k$}} \\
    \quad \\
    \text{Level 2} \\
    \mu^{(k)}        &= \mu + \xi^{(k)} &\quad\small{\text{Average weight of litter $k$}} \\
    \alpha^{(k)}_{j} &= \alpha_{j} &\quad\small{\text{Effect of sex $j$ in litter $k$}}
\end{alignat*}
$$

with

$$
\begin{alignat*}{1}
    \eta^{(k)}_{ij} &\sim \mathcal{N}(0,\sigma^{2}_{1}) \\
    \xi^{(k)}       &\sim \mathcal{N}(0,\sigma^{2}_{2}).
\end{alignat*}
$$

So, there are only *two* error terms now, rather than *three*. Choosing to make something *fixed* rather than *random* is equivalent to *removing* its random error term. If we then continue with our logic of building the Level 2 models, we can still include `treatment` and `sex:treatment` and end up with 

$$
\begin{alignat*}{1}
    \text{Level 1} \\
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha^{(k)}_{j} + \eta^{(k)}_{ij}  \\
    \quad \\
    \text{Level 2} \\
    \mu^{(k)}        &= \mu + \beta_{l} + \xi^{(k)}  \\
    \alpha^{(k)}_{j} &= \alpha_{j} + (\alpha\beta)_{jl}.
\end{alignat*}
$$

We can even choose to collapse the $\alpha^{(k)}_{j}$ effects into Level 1. This makes it clear that there are *no* litter-specific effects of `sex` anymore and there is only a single random-effect that has a complete Level 2 model.

$$
\begin{alignat*}{1}
    \text{Level 1} \\
    y^{(k)}_{ij}     &= \mu^{(k)} + \alpha_{j} + (\alpha\beta)_{jl} + \eta^{(k)}_{ij}  \\
    \quad \\
    \text{Level 2} \\
    \mu^{(k)}        &= \mu + \beta_{l} + \xi^{(k)}.
\end{alignat*}
$$

More intuition about treating `sex` as fixed or random is given in the drop-down below.


````{admonition} More on Random vs Fixed
:class: tip, dropdown
As discussed above, the only structural difference between treating `sex` as random or fixed is the inclusion or absence of an error term for the model of $\alpha^{(k)}_{j}$. This is because the errors are *where* the randomness lives. Although this seems like a simple change, it actually has quite a big effect on our assumptions about the data-generating process. In general, our choice comes down to:~

- Fixed effect of sex
    - Every litter obeys the same biological rule. Males are heavier than females by the same amount.
- Random effect of sex
    - The male–female difference depends on the litter. In some litters males are much heavier, in others only slightly heavier, and in some not at all.

In both models the general structure is the same, however our assumptions about the parameters determine whether we include an additional error term. If we do include one, this captures the idea that each $\alpha^{(k)}_{j}$ is a random draw from a population of possible $\alpha^{(k)}_{j}$ effects with a mean given by $\alpha_{j} + (\alpha\beta)_{jl}$. Under this view, even if we knew the values of $\alpha_{j}$ and $(\alpha\beta)_{jl}$, we still could not predict each $\alpha^{(k)}_{j}$ exactly. Instead, those parameters describe the average effect of `sex` across litters, and each individual litter deviates from that average. The additional error term captures this *between-litter* variability in the effect of `sex`.

If we do not include an error term, this captures a different assumption. In this case each $\alpha^{(k)}_{j}$ is completely determined by $\alpha_{j}$ and $(\alpha\beta)_{jl}$. If we knew those parameters exactly, we would know exactly how `sex` influences `weight` in every litter. Any differences we observe between litters would therefore arise only from sampling variation, not from genuine differences in the underlying biological effect. Under this assumption, the effect of `sex` does not vary from litter to litter. It is a single population value that applies equally to all litters. In that sense $\alpha^{(k)}_{j}$ is not a random draw from a distribution of possible effects, but a fixed value that would be known exactly if the entire population were observed.
````

[^litsize-foot]: You can consider it an additional challenge to think about how `litsize` could fit into all this. 